In [0]:
pip install polars lets_plot xgboost shap rapidfuzz

In [0]:
%restart_python

In [0]:
# ── IMPORTS ───────────────────────────────────────────────────────────────────
import polars as pl
import pandas as pd
import numpy as np
from rapidfuzz import process, fuzz
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
import xgboost as xgb
import itertools
import warnings
warnings.filterwarnings("ignore")

# ── LOAD DATA ─────────────────────────────────────────────────────────────────
pst        = pl.read_csv("march_madness_post.csv")
df_2026    = pl.read_csv("2026_team_results.csv")
mteams     = pl.read_csv("MTeams.csv")
spellings  = pl.read_csv("MTeamSpellings.csv")
submission = pl.read_csv("SampleSubmissionStage2.csv")

print(f"PST rows: {len(pst)}, Seasons: {pst['Season'].n_unique()}")
print(f"2026 teams: {len(df_2026)}")
print(f"Submission rows: {len(submission)}")

# ── STEP 1: FUZZY NAME MAPPING (barttorvik short → pst full name) ─────────────
bart_teams = df_2026.select('team').to_series().to_list()
pst_teams  = pst.filter(pl.col('Season') == 2025).select('Full Team Name').to_series().to_list()

name_map = {}
for bt in bart_teams:
    match, score, _ = process.extractOne(bt, pst_teams, scorer=fuzz.token_sort_ratio)
    name_map[bt] = match if score >= 60 else None

# Manual overrides for known mismatches
manual_overrides = {
    "UConn":             "Connecticut Huskies",
    "N.C. State":        "NC State Wolfpack",
    "Miami FL":          "Miami Hurricanes",
    "UIW":               "Incarnate Word Cardinals",
    "UTRGV":             "UT Rio Grande Valley Vaqueros",
    "TAM C. Christi":    "Texas A&M-Corpus Christi Islanders",
    "St. John's":        "St. John's Red Storm",
    "Saint Mary's":      "Saint Mary's Gaels",
    "VCU":               "VCU Rams",
    "UCF":               "UCF Knights",
    "UNLV":              "UNLV Rebels",
    "UAB":               "UAB Blazers",
    "FIU":               "FIU Panthers",
    "UTEP":              "UTEP Miners",
    "UTSA":              "UTSA Roadrunners",
    "LIU":               "LIU Sharks",
    "SIUE":              "SIU Edwardsville Cougars",
    "SMU":               "SMU Mustangs",
    "TCU":               "TCU Horned Frogs",
    "BYU":               "BYU Cougars",
    "LSU":               "LSU Tigers",
    "McNeese":           "McNeese State Cowboys",
    "Miami OH":          "Miami (OH) RedHawks",
    "USC":               "USC Trojans",
    "Pitt":              "Pittsburgh Panthers",
    "CSUN":              "CS Northridge Matadors",
    "UNC Wilmington":    "UNC Wilmington Seahawks",
    "Utah Valley":       "Utah Valley Wolverines",
    "Stephen F. Austin": "Stephen F. Austin Lumberjacks",
    "High Point":        "High Point Panthers",
}
name_map.update(manual_overrides)

unmatched = [k for k, v in name_map.items() if v is None]
print(f"\nUnmatched teams: {len(unmatched)}")
if unmatched:
    print(unmatched)

# ── STEP 2: DEFINE FEATURES ───────────────────────────────────────────────────
PST_FEATURES = [f for f in [
    'AdjEM', 'AdjOE', 'AdjDE', 'AdjTempo',
    'Net Rating', 'eFGPct', 'TOPct', 'ORPct', 'FTRate',
    'FG3Pct', 'FG3Rate', 'BlockPct', 'OppFG3Pct',
    'OppBlockPct', 'OppStlRate', 'StlRate',
    'Experience', 'AvgHeight', 'EffectiveHeight',
    'Active Coaching Length Index',
] if f in pst.columns]
print(f"\nFeatures available: {len(PST_FEATURES)}: {PST_FEATURES}")

# ── STEP 3: FILTER TOURNAMENT TEAMS ──────────────────────────────────────────
tourney_teams = pst.filter(
    pl.col('Post-Season Tournament') == 'March Madness'
).select(['Season', 'Full Team Name'] + PST_FEATURES +
         ['Tournament Winner?', 'Final Four?', 'Tournament Championship?'])

print(f"\nTournament team-seasons: {len(tourney_teams)}")

# ── STEP 4: BUILD PAIRWISE TRAINING DATA ─────────────────────────────────────
records = []
seasons = tourney_teams.select('Season').unique().to_series().to_list()
skipped = 0

for season in sorted(seasons):
    season_teams = tourney_teams.filter(pl.col('Season') == season).to_pandas()
    season_teams = season_teams.drop_duplicates(subset='Full Team Name')

    champs  = season_teams[season_teams['Tournament Championship?'] == 'Yes']['Full Team Name'].tolist()
    winners = season_teams[season_teams['Tournament Winner?'] == 'Yes']['Full Team Name'].tolist()
    teams   = season_teams['Full Team Name'].tolist()
    lookup  = {row['Full Team Name']: row for _, row in season_teams.iterrows()}

    for t1, t2 in itertools.combinations(teams, 2):
        if t1 not in lookup or t2 not in lookup:
            skipped += 1
            continue

        row1 = lookup[t1]
        row2 = lookup[t2]

        if   t1 in champs:   label = 1
        elif t2 in champs:   label = 0
        elif t1 in winners:  label = 1
        elif t2 in winners:  label = 0
        else: label = 1 if row1['AdjEM'] > row2['AdjEM'] else 0

        diff = {}
        for f in PST_FEATURES:
            try:    diff[f'diff_{f}'] = float(row1[f]) - float(row2[f])
            except: diff[f'diff_{f}'] = 0.0
        diff['label'] = label
        records.append(diff)

train_df    = pl.DataFrame(records)
feature_cols = [c for c in train_df.columns if c.startswith('diff_')]
print(f"Training pairs: {len(train_df)}, Features: {len(feature_cols)}, Skipped: {skipped}")

# ── STEP 5: TRAIN MODEL ───────────────────────────────────────────────────────
train_pd = train_df.to_pandas()
X = train_pd[feature_cols].fillna(0).values
y = train_pd['label'].values

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

lr = LogisticRegression(C=0.1, max_iter=1000, random_state=42)
lr_cv = cross_val_score(lr, X_scaled, y, cv=5, scoring='neg_log_loss')
print(f"\nLogistic Regression CV Log Loss: {-lr_cv.mean():.4f} ± {lr_cv.std():.4f}")

xgb_model = xgb.XGBClassifier(
    n_estimators=200, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='logloss', random_state=42
)
xgb_cv = cross_val_score(xgb_model, X, y, cv=5, scoring='neg_log_loss')
print(f"XGBoost CV Log Loss:             {-xgb_cv.mean():.4f} ± {xgb_cv.std():.4f}")

xgb_model.fit(X, y)
lr.fit(X_scaled, y)
print("Models trained!")

# ── STEP 6: BUILD 2026 FEATURE LOOKUP ────────────────────────────────────────
pst_2025_avg = pst.filter(pl.col('Season') == 2025).select(PST_FEATURES).mean().to_pandas().iloc[0]

team_features_2026 = {}
for row in df_2026.to_pandas().itertuples():
    pst_name = name_map.get(row.team)
    if pst_name is None:
        continue
    feats = {f: pst_2025_avg[f] for f in PST_FEATURES}  # defaults
    feats['AdjOE']    = row.adjoe
    feats['AdjDE']    = row.adjde
    feats['AdjTempo'] = row.adjt
    feats['AdjEM']    = row.adjoe - row.adjde
    feats['Net Rating'] = row.barthag * 100
    team_features_2026[pst_name] = feats

print(f"\n2026 teams with features: {len(team_features_2026)}")

# ── STEP 7: MAP PST NAMES → KAGGLE TEAM IDs ──────────────────────────────────
spell_lookup = {}
for row in spellings.to_pandas().itertuples():
    spell_lookup[row.TeamNameSpelling.lower().strip()] = row.TeamID

def name_to_id(full_name):
    key = full_name.lower().strip()
    if key in spell_lookup:
        return spell_lookup[key]
    match, score, _ = process.extractOne(key, list(spell_lookup.keys()), scorer=fuzz.token_sort_ratio)
    return spell_lookup[match] if score >= 75 else None

team_id_features = {}
unmapped = []
for pst_name, feats in team_features_2026.items():
    tid = name_to_id(pst_name)
    if tid:
        team_id_features[tid] = feats
    else:
        unmapped.append(pst_name)

print(f"Teams mapped to Kaggle TeamID: {len(team_id_features)}")
if unmapped:
    print(f"WARNING - no TeamID found for: {unmapped}")

# ── STEP 8: GENERATE PREDICTIONS ─────────────────────────────────────────────
predictions = []
missing     = []

for row in submission.to_pandas().itertuples():
    parts = row.ID.split('_')
    team_a, team_b = int(parts[1]), int(parts[2])

    if team_a in team_id_features and team_b in team_id_features:
        fa   = team_id_features[team_a]
        fb   = team_id_features[team_b]
        diff = np.array([
            fa.get(f.replace('diff_', ''), 0) - fb.get(f.replace('diff_', ''), 0)
            for f in feature_cols
        ]).reshape(1, -1)

        prob = xgb_model.predict_proba(diff)[0][1]
        prob = np.clip(prob, 0.025, 0.975)
    else:
        missing.append(row.ID)
        prob = 0.5

    predictions.append({'ID': row.ID, 'Pred': prob})

print(f"\nPredictions generated: {len(predictions)}")
print(f"Defaulted to 0.5: {len(missing)}")

# ── STEP 9: SAVE SUBMISSION ───────────────────────────────────────────────────
submission_out = pl.DataFrame(predictions)
submission_out.write_csv("submission_2026.csv")

print("\nPrediction distribution:")
print(submission_out.select('Pred').describe())
print("\nSample predictions:")
print(submission_out.head(10))
print("\n✓ Saved submission_2026.csv — ready to upload to Kaggle!")

In [0]:
seeds_2026 = pl.read_csv("MNCAATourneySeeds.csv")
print(seeds_2026.filter(pl.col('Season') == 2026))

In [0]:
rankings_2026 = df_2026.select(['team', 'adjoe', 'adjde', 'barthag', 'WAB']).to_pandas()
rankings_2026['AdjEM'] = rankings_2026['adjoe'] - rankings_2026['adjde']
rankings_2026 = rankings_2026.sort_values('AdjEM', ascending=False).reset_index(drop=True)
print(rankings_2026[['team', 'AdjEM', 'barthag']].head(30))

In [0]:
# ── BRACKET SIMULATOR ────────────────────────────────────────────────────────

# Step 1: Load seeds and team names
seeds_2026 = pl.read_csv("MNCAATourneySeeds.csv").filter(pl.col('Season') == 2026)
mteams = pl.read_csv("MTeams.csv")

# Join seeds with team names
bracket = seeds_2026.join(mteams.select(['TeamID', 'TeamName']), on='TeamID', how='left').to_pandas()
bracket['Region'] = bracket['Seed'].str[0]
bracket['SeedNum'] = bracket['Seed'].str[1:3].astype(int)

# Step 2: Join AdjEM ratings from barttorvik
rankings_2026 = df_2026.select(['team', 'adjoe', 'adjde', 'barthag', 'WAB']).to_pandas()
rankings_2026['AdjEM'] = rankings_2026['adjoe'] - rankings_2026['adjde']

# Fuzzy match TeamName to barttorvik name to get ratings
from rapidfuzz import process, fuzz

def get_adjEM(team_name):
    match, score, _ = process.extractOne(team_name, rankings_2026['team'].tolist(), scorer=fuzz.token_sort_ratio)
    if score >= 60:
        return rankings_2026[rankings_2026['team'] == match]['AdjEM'].values[0]
    return 0.0

def get_barthag(team_name):
    match, score, _ = process.extractOne(team_name, rankings_2026['team'].tolist(), scorer=fuzz.token_sort_ratio)
    if score >= 60:
        return rankings_2026[rankings_2026['team'] == match]['barthag'].values[0]
    return 0.5

bracket['AdjEM']   = bracket['TeamName'].apply(get_adjEM)
bracket['barthag'] = bracket['TeamName'].apply(get_barthag)

print("Bracket teams loaded:")
print(bracket[['Seed', 'TeamName', 'AdjEM', 'barthag']].sort_values('Seed').to_string())

In [0]:
# ── ROUND BY ROUND BRACKET PREDICTION ────────────────────────────────────────

def predict_winner(team_a, team_b):
    """Pick winner based on AdjEM. Returns winning row."""
    return team_a if team_a['AdjEM'] > team_b['AdjEM'] else team_b

def simulate_region(region_teams, region_name):
    """Simulate one region through Elite Eight."""
    # Sort by seed number so matchups are correct (1v16, 2v15, etc.)
    teams = region_teams.sort_values('SeedNum').reset_index(drop=True)
    
    round_num = 1
    print(f"\n{'='*50}")
    print(f"  REGION: {region_name}")
    print(f"{'='*50}")
    
    while len(teams) > 1:
        round_name = {1: 'Round of 64', 2: 'Round of 32', 
                      3: 'Sweet 16', 4: 'Elite Eight'}[round_num]
        print(f"\n--- {round_name} ---")
        
        winners = []
        # Pair first vs last, second vs second-to-last, etc.
        n = len(teams)
        for i in range(n // 2):
            t1 = teams.iloc[i]
            t2 = teams.iloc[n - 1 - i]
            winner = predict_winner(t1, t2)
            loser  = t2 if winner['TeamName'] == t1['TeamName'] else t1
            print(f"  ({int(t1['SeedNum'])}) {t1['TeamName']:20s} vs ({int(t2['SeedNum'])}) {t2['TeamName']:20s}  →  ✓ {winner['TeamName']}")
            winners.append(winner)
        
        teams = pd.DataFrame(winners).reset_index(drop=True)
        round_num += 1
    
    print(f"\n  🏆 {region_name} Champion: {teams.iloc[0]['TeamName']}")
    return teams.iloc[0]

# ── RUN ALL 4 REGIONS ─────────────────────────────────────────────────────────
import pandas as pd

region_map = {'W': 'West', 'X': 'East', 'Y': 'South', 'Z': 'Midwest'}
final_four = []

for region_code, region_name in region_map.items():
    region_teams = bracket[bracket['Region'] == region_code].copy()
    # Remove play-in teams (seed like W16a, W16b) - keep only clean seeds
    region_teams = region_teams[~bracket['Seed'].str.contains('a|b')].copy()
    champion = simulate_region(region_teams, region_name)
    final_four.append(champion)

# ── FINAL FOUR ────────────────────────────────────────────────────────────────
print(f"\n{'='*50}")
print("  FINAL FOUR")
print(f"{'='*50}")

# W vs X, Y vs Z
ff_teams = pd.DataFrame(final_four)

sf1_teams = ff_teams[ff_teams['Region'].isin(['W', 'X'])]
sf2_teams = ff_teams[ff_teams['Region'].isin(['Y', 'Z'])]

t1, t2 = sf1_teams.iloc[0], sf1_teams.iloc[1]
winner1 = predict_winner(t1, t2)
print(f"\n  {t1['TeamName']} vs {t2['TeamName']}  →  ✓ {winner1['TeamName']}")

t3, t4 = sf2_teams.iloc[0], sf2_teams.iloc[1]
winner2 = predict_winner(t3, t4)
print(f"  {t3['TeamName']} vs {t4['TeamName']}  →  ✓ {winner2['TeamName']}")

# ── CHAMPIONSHIP ──────────────────────────────────────────────────────────────
print(f"\n{'='*50}")
print("  NATIONAL CHAMPIONSHIP")
print(f"{'='*50}")
champion = predict_winner(winner1, winner2)
print(f"\n  {winner1['TeamName']} vs {winner2['TeamName']}")
print(f"\n  🏆 2026 NATIONAL CHAMPION: {champion['TeamName']} 🏆")

In [0]:
# Check what seeds we have per region
for region in ['W', 'X', 'Y', 'Z']:
    region_teams = bracket[bracket['Region'] == region].copy()
    seeds_present = sorted(region_teams['SeedNum'].tolist())
    print(f"Region {region}: {len(region_teams)} teams, seeds: {seeds_present}")
    
    # Show any with a/b in seed
    playin = bracket[bracket['Seed'].str.contains('a|b', regex=True)]
    print(f"  Play-in teams: {playin[['Seed','TeamName']].values.tolist()}")

In [0]:
print(f"Total bracket teams: {len(bracket)}")
print(bracket[['Seed', 'TeamName', 'AdjEM']].sort_values('Seed').to_string())

In [0]:
import pandas as pd
import numpy as np
from rapidfuzz import process, fuzz

def predict_winner(t1, t2):
    return t1 if t1['AdjEM'] > t2['AdjEM'] else t2

# ── STEP 1: RESOLVE PLAY-IN GAMES ────────────────────────────────────────────
print("="*50)
print("  FIRST FOUR (PLAY-IN GAMES)")
print("="*50)

bracket_pd = bracket.copy()

def play_in(seed_a, seed_b, winner_seed):
    t1 = bracket_pd[bracket_pd['Seed'] == seed_a].iloc[0]
    t2 = bracket_pd[bracket_pd['Seed'] == seed_b].iloc[0]
    winner = predict_winner(t1, t2)
    loser  = t2 if winner['TeamName'] == t1['TeamName'] else t1
    print(f"  {t1['TeamName']:20s} vs {t2['TeamName']:20s}  →  ✓ {winner['TeamName']}")
    # Add winner back as the clean seed, remove both play-in rows
    new_row = winner.copy()
    new_row['Seed']    = winner_seed
    new_row['SeedNum'] = int(winner_seed[1:])
    return new_row, [seed_a, seed_b]

new_rows = []
remove_seeds = []

r1, r = play_in('X16a', 'X16b', 'X16'); new_rows.append(r1); remove_seeds += r
r2, r = play_in('Y16a', 'Y16b', 'Y16'); new_rows.append(r2); remove_seeds += r
r3, r = play_in('Y11a', 'Y11b', 'Y11'); new_rows.append(r3); remove_seeds += r
r4, r = play_in('Z11a', 'Z11b', 'Z11'); new_rows.append(r4); remove_seeds += r

# Remove play-in entries and add winners
bracket_clean = bracket_pd[~bracket_pd['Seed'].isin(remove_seeds)].copy()
bracket_clean = pd.concat([bracket_clean, pd.DataFrame(new_rows)], ignore_index=True)

# Verify 16 teams per region
for region in ['W', 'X', 'Y', 'Z']:
    teams = bracket_clean[bracket_clean['Region'] == region]
    print(f"\nRegion {region}: {len(teams)} teams ✓" if len(teams)==16 else f"\nRegion {region}: {len(teams)} teams ✗ CHECK THIS")

# ── STEP 2: SIMULATE REGIONS ─────────────────────────────────────────────────

def simulate_region(region_code, region_name):
    teams = bracket_clean[bracket_clean['Region'] == region_code].sort_values('SeedNum').reset_index(drop=True)
    
    print(f"\n{'='*55}")
    print(f"  REGION: {region_name}")
    print(f"{'='*55}")
    
    round_names = {1: 'Round of 64', 2: 'Round of 32', 3: 'Sweet 16', 4: 'Elite Eight'}
    round_num = 1
    
    while len(teams) > 1:
        print(f"\n  --- {round_names[round_num]} ---")
        winners = []
        n = len(teams)
        for i in range(n // 2):
            t1 = teams.iloc[i]
            t2 = teams.iloc[n - 1 - i]
            winner = predict_winner(t1, t2)
            print(f"  ({int(t1['SeedNum'])}) {t1['TeamName']:22s} vs ({int(t2['SeedNum'])}) {t2['TeamName']:22s}  →  ✓ {winner['TeamName']}")
            winners.append(winner)
        teams = pd.DataFrame(winners).reset_index(drop=True)
        round_num += 1
    
    print(f"\n  🏆 {region_name} Champion: {teams.iloc[0]['TeamName']}")
    return teams.iloc[0]

region_map = {'W': 'West', 'X': 'East', 'Y': 'South', 'Z': 'Midwest'}
final_four = []
for code, name in region_map.items():
    winner = simulate_region(code, name)
    final_four.append(winner)

# ── STEP 3: FINAL FOUR ────────────────────────────────────────────────────────
print(f"\n{'='*55}")
print("  FINAL FOUR")
print(f"{'='*55}")

ff = pd.DataFrame(final_four)

# W vs X, Y vs Z
w = ff[ff['Region'] == 'W'].iloc[0]
x = ff[ff['Region'] == 'X'].iloc[0]
y = ff[ff['Region'] == 'Y'].iloc[0]
z = ff[ff['Region'] == 'Z'].iloc[0]

sf1 = predict_winner(w, x)
print(f"\n  (West)    {w['TeamName']:20s} vs (East)    {x['TeamName']:20s}  →  ✓ {sf1['TeamName']}")

sf2 = predict_winner(y, z)
print(f"  (South)   {y['TeamName']:20s} vs (Midwest) {z['TeamName']:20s}  →  ✓ {sf2['TeamName']}")

# ── STEP 4: CHAMPIONSHIP ─────────────────────────────────────────────────────
print(f"\n{'='*55}")
print("  NATIONAL CHAMPIONSHIP")
print(f"{'='*55}")

champion = predict_winner(sf1, sf2)
print(f"\n  {sf1['TeamName']:20s} vs {sf2['TeamName']}")
print(f"\n  🏆 2026 NATIONAL CHAMPION: {champion['TeamName']} 🏆")

In [0]:
# ── USE THE TRAINED XGB MODEL FOR BRACKET PREDICTIONS ────────────────────────
# Instead of just comparing AdjEM, we feed feature differences into xgb_model

def get_team_features(team_name):
    """Get full feature vector for a team from pst 2025 data as proxy for 2026."""
    # First try to find in team_features_2026 via name mapping
    pst_name = name_map.get(team_name)
    
    if pst_name and pst_name in team_features_2026:
        return team_features_2026[pst_name]
    
    # Fallback: fuzzy match directly against pst 2025
    pst_2025 = pst.filter(pl.col('Season') == 2025).to_pandas()
    match, score, _ = process.extractOne(
        team_name, 
        pst_2025['Full Team Name'].tolist(), 
        scorer=fuzz.token_sort_ratio
    )
    if score >= 60:
        row = pst_2025[pst_2025['Full Team Name'] == match].iloc[0]
        return {f: row[f] for f in PST_FEATURES}
    
    # Last resort: use AdjEM from barttorvik only
    bart_match, score, _ = process.extractOne(
        team_name, 
        rankings_2026['team'].tolist(), 
        scorer=fuzz.token_sort_ratio
    )
    if score >= 60:
        bart_row = rankings_2026[rankings_2026['team'] == bart_match].iloc[0]
        feats = {f: pst_2025_avg[f] for f in PST_FEATURES}
        feats['AdjEM']    = bart_row['AdjEM']
        feats['AdjOE']    = bart_row['adjoe']
        feats['AdjDE']    = bart_row['adjde']
        feats['Net Rating'] = bart_row['barthag'] * 100
        return feats
    
    return {f: 0.0 for f in PST_FEATURES}

def ml_predict_winner(t1, t2):
    """Use trained XGBoost model to predict winner. Returns winning row + probability."""
    f1 = get_team_features(t1['TeamName'])
    f2 = get_team_features(t2['TeamName'])
    
    diff = np.array([
        f1.get(f.replace('diff_', ''), 0) - f2.get(f.replace('diff_', ''), 0)
        for f in feature_cols
    ]).reshape(1, -1)
    
    prob_t1_wins = xgb_model.predict_proba(diff)[0][1]
    
    if prob_t1_wins >= 0.5:
        return t1, prob_t1_wins
    else:
        return t2, 1 - prob_t1_wins

# ── ML BRACKET SIMULATOR ──────────────────────────────────────────────────────

def simulate_region_ml(region_code, region_name):
    teams = bracket_clean[bracket_clean['Region'] == region_code].sort_values('SeedNum').reset_index(drop=True)
    
    print(f"\n{'='*60}")
    print(f"  REGION: {region_name}")
    print(f"{'='*60}")
    
    round_names = {1: 'Round of 64', 2: 'Round of 32', 3: 'Sweet 16', 4: 'Elite Eight'}
    round_num = 1
    
    while len(teams) > 1:
        print(f"\n  --- {round_names[round_num]} ---")
        winners = []
        n = len(teams)
        for i in range(n // 2):
            t1 = teams.iloc[i]
            t2 = teams.iloc[n - 1 - i]
            winner, prob = ml_predict_winner(t1, t2)
            loser = t2 if winner['TeamName'] == t1['TeamName'] else t1
            print(f"  ({int(t1['SeedNum'])}) {t1['TeamName']:22s} vs ({int(t2['SeedNum'])}) {t2['TeamName']:22s}  →  ✓ {winner['TeamName']:22s} ({prob:.0%})")
            winners.append(winner)
        
        teams = pd.DataFrame(winners).reset_index(drop=True)
        round_num += 1
    
    print(f"\n  🏆 {region_name} Champion: {teams.iloc[0]['TeamName']}")
    return teams.iloc[0]

# ── RUN ML BRACKET ────────────────────────────────────────────────────────────
region_map = {'W': 'West', 'X': 'East', 'Y': 'South', 'Z': 'Midwest'}
final_four_ml = []

for code, name in region_map.items():
    winner = simulate_region_ml(code, name)
    final_four_ml.append(winner)

# ── FINAL FOUR ────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("  FINAL FOUR")
print(f"{'='*60}")

ff = pd.DataFrame(final_four_ml)
w = ff[ff['Region'] == 'W'].iloc[0]
x = ff[ff['Region'] == 'X'].iloc[0]
y = ff[ff['Region'] == 'Y'].iloc[0]
z = ff[ff['Region'] == 'Z'].iloc[0]

sf1, p1 = ml_predict_winner(w, x)
print(f"\n  (West)    {w['TeamName']:20s} vs (East)    {x['TeamName']:20s}  →  ✓ {sf1['TeamName']} ({p1:.0%})")

sf2, p2 = ml_predict_winner(y, z)
print(f"  (South)   {y['TeamName']:20s} vs (Midwest) {z['TeamName']:20s}  →  ✓ {sf2['TeamName']} ({p2:.0%})")

# ── CHAMPIONSHIP ──────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("  NATIONAL CHAMPIONSHIP")
print(f"{'='*60}")

champion, p_champ = ml_predict_winner(sf1, sf2)
print(f"\n  {sf1['TeamName']:20s} vs {sf2['TeamName']}")
print(f"\n  🏆 2026 NATIONAL CHAMPION: {champion['TeamName']} ({p_champ:.0%} confidence) 🏆")

In [0]:
# Check they loaded correctly
tourney_results = pl.read_csv("MNCAATourneyCompactResults.csv")
massey = pl.read_csv("MMasseyOrdinals.csv")

print(tourney_results.filter(pl.col('Season') >= 2020).head(5))
print(f"\nMassey systems available: {massey['SystemName'].n_unique()}")
print(f"Massey seasons: {sorted(massey['Season'].unique().to_list())}")

In [0]:
# ── IMPROVED MODEL WITH REAL MATCHUPS + MASSEY RANKINGS ──────────────────────

import polars as pl
import pandas as pd
import numpy as np
from rapidfuzz import process, fuzz
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
import xgboost as xgb
import warnings
warnings.filterwarnings("ignore")

# ── LOAD ALL DATA ─────────────────────────────────────────────────────────────
pst             = pl.read_csv("march_madness_post.csv")
df_2026         = pl.read_csv("2026_team_results.csv")
mteams          = pl.read_csv("MTeams.csv")
spellings       = pl.read_csv("MTeamSpellings.csv")
submission      = pl.read_csv("SampleSubmissionStage2.csv")
tourney_results = pl.read_csv("MNCAATourneyCompactResults.csv")
massey          = pl.read_csv("MMasseyOrdinals.csv")
seeds_df        = pl.read_csv("MNCAATourneySeeds.csv")

print("All files loaded!")

# ── STEP 1: PICK BEST MASSEY SYSTEMS ─────────────────────────────────────────
# Use pre-tournament rankings (RankingDayNum=133) from most predictive systems
# These are well-known systems that correlate strongly with tournament outcomes
BEST_SYSTEMS = ['POM', 'SAG', 'MOR', 'DOL', 'RPI', 'WLK', 'BPI', 'NET', 'KPI', 'KEN']

# Keep only systems that actually exist in the data
available_systems = massey.select('SystemName').unique().to_series().to_list()
BEST_SYSTEMS = [s for s in BEST_SYSTEMS if s in available_systems]
print(f"Using Massey systems: {BEST_SYSTEMS}")

# Get pre-tournament rankings only (RankingDayNum=133)
massey_pretourney = massey.filter(
    (pl.col('RankingDayNum') == 133) &
    (pl.col('SystemName').is_in(BEST_SYSTEMS))
)
print(f"Pre-tournament ranking rows: {len(massey_pretourney)}")

# Pivot to wide format: one row per (Season, TeamID), one col per system
massey_wide = massey_pretourney.to_pandas().pivot_table(
    index=['Season', 'TeamID'],
    columns='SystemName',
    values='OrdinalRank',
    aggfunc='first'
).reset_index()
massey_wide.columns = ['Season', 'TeamID'] + [f'Massey_{s}' for s in massey_wide.columns[2:]]
massey_wide = massey_wide.fillna(200)  # fill missing with median-ish rank
print(f"Massey wide shape: {massey_wide.shape}")
print(f"Massey features: {[c for c in massey_wide.columns if c.startswith('Massey_')]}")

# ── STEP 2: ADD SEED AS FEATURE ───────────────────────────────────────────────
seeds_clean = seeds_df.to_pandas()
seeds_clean['SeedNum'] = seeds_clean['Seed'].str[1:3].astype(int)
seed_lookup = seeds_clean.set_index(['Season', 'TeamID'])['SeedNum']

# ── STEP 3: BUILD FEATURE LOOKUP PER TEAM PER SEASON ─────────────────────────
# Merge massey rankings with pst stats using TeamID

# Build TeamID → pst name mapping via spellings
spell_pd = spellings.to_pandas()
spell_lookup = {row['TeamNameSpelling'].lower().strip(): row['TeamID'] 
                for _, row in spell_pd.iterrows()}

def name_to_id(full_name):
    if full_name is None or pd.isna(full_name):
        return None
    key = str(full_name).lower().strip()
    if key in spell_lookup:
        return spell_lookup[key]
    match, score, _ = process.extractOne(key, list(spell_lookup.keys()), scorer=fuzz.token_sort_ratio)
    return spell_lookup[match] if score >= 75 else None

# Add TeamID to pst
pst_pd = pst.to_pandas()
pst_pd['TeamID'] = pst_pd['Full Team Name'].apply(name_to_id)
print(f"PST rows with TeamID mapped: {pst_pd['TeamID'].notna().sum()} / {len(pst_pd)}")

PST_FEATURES = [f for f in [
    'AdjEM', 'AdjOE', 'AdjDE', 'AdjTempo', 'Net Rating',
    'eFGPct', 'TOPct', 'ORPct', 'FTRate', 'FG3Pct', 'FG3Rate',
    'BlockPct', 'OppFG3Pct', 'OppBlockPct', 'OppStlRate', 'StlRate',
    'Experience', 'AvgHeight', 'EffectiveHeight', 'Active Coaching Length Index',
] if f in pst_pd.columns]

MASSEY_FEATURES = [c for c in massey_wide.columns if c.startswith('Massey_')]
ALL_FEATURES = PST_FEATURES + MASSEY_FEATURES + ['SeedNum']
print(f"\nTotal features: {len(ALL_FEATURES)}")
print(f"  PST: {len(PST_FEATURES)}, Massey: {len(MASSEY_FEATURES)}, Seed: 1")

# ── STEP 4: BUILD TEAM FEATURE DICT (Season, TeamID) → features ──────────────
# Merge pst + massey
pst_massey = pst_pd.merge(
    massey_wide, 
    on=['Season', 'TeamID'], 
    how='left'
).fillna(0)

team_feat_lookup = {}
for _, row in pst_massey.iterrows():
    key = (int(row['Season']), int(row['TeamID'])) if pd.notna(row['TeamID']) else None
    if key:
        team_feat_lookup[key] = {f: row.get(f, 0) for f in ALL_FEATURES}

print(f"Team-season feature vectors built: {len(team_feat_lookup)}")

# ── STEP 5: BUILD TRAINING DATA FROM REAL TOURNAMENT GAMES ───────────────────
tourney_pd = tourney_results.to_pandas()
tourney_pd = tourney_pd[tourney_pd['Season'] >= 2003]  # match massey coverage

records = []
skipped = 0

for _, row in tourney_pd.iterrows():
    season   = int(row['Season'])
    w_team   = int(row['WTeamID'])
    l_team   = int(row['LTeamID'])
    
    # Always put lower TeamID as "team_a"
    team_a = min(w_team, l_team)
    team_b = max(w_team, l_team)
    label  = 1 if w_team == team_a else 0
    
    key_a = (season, team_a)
    key_b = (season, team_b)
    
    if key_a not in team_feat_lookup or key_b not in team_feat_lookup:
        skipped += 1
        continue
    
    fa = team_feat_lookup[key_a]
    fb = team_feat_lookup[key_b]
    
    # Add seed difference
    try:
        fa['SeedNum'] = seed_lookup.loc[(season, team_a)]
        fb['SeedNum'] = seed_lookup.loc[(season, team_b)]
    except:
        fa['SeedNum'] = 8
        fb['SeedNum'] = 8
    
    diff = {f'diff_{f}': fa.get(f, 0) - fb.get(f, 0) for f in ALL_FEATURES}
    diff['label'] = label
    records.append(diff)

train_df    = pd.DataFrame(records)
feature_cols = [c for c in train_df.columns if c.startswith('diff_')]
print(f"\nReal tournament matchups: {len(train_df)}, Skipped: {skipped}")
print(f"Feature columns: {len(feature_cols)}")

# ── STEP 6: TRAIN IMPROVED MODEL ─────────────────────────────────────────────
X = train_df[feature_cols].fillna(0).values
y = train_df['label'].values

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

lr = LogisticRegression(C=0.1, max_iter=1000, random_state=42)
lr_cv = cross_val_score(lr, X_scaled, y, cv=5, scoring='neg_log_loss')
print(f"\nLogistic Regression CV Log Loss: {-lr_cv.mean():.4f} ± {lr_cv.std():.4f}")

xgb_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='logloss', random_state=42
)
xgb_cv = cross_val_score(xgb_model, X, y, cv=5, scoring='neg_log_loss')
print(f"XGBoost CV Log Loss:             {-xgb_cv.mean():.4f} ± {xgb_cv.std():.4f}")

xgb_model.fit(X, y)
lr.fit(X_scaled, y)
print("\nModels trained!")

# ── STEP 7: BUILD 2026 FEATURE LOOKUP ────────────────────────────────────────
# Get 2026 Massey rankings
massey_2026 = massey.filter(
    (pl.col('Season') == 2026) &
    (pl.col('RankingDayNum') == 133) &
    (pl.col('SystemName').is_in(BEST_SYSTEMS))
).to_pandas().pivot_table(
    index='TeamID',
    columns='SystemName',
    values='OrdinalRank',
    aggfunc='first'
).reset_index()
massey_2026.columns = ['TeamID'] + [f'Massey_{s}' for s in massey_2026.columns[1:]]
massey_2026 = massey_2026.fillna(200)
print(f"\n2026 Massey teams: {len(massey_2026)}")

# Get 2026 pst stats (use barttorvik as proxy)
bart_pd = df_2026.to_pandas()
bart_pd['AdjEM']    = bart_pd['adjoe'] - bart_pd['adjde']
bart_pd['AdjOE']    = bart_pd['adjoe']
bart_pd['AdjDE']    = bart_pd['adjde']
bart_pd['AdjTempo'] = bart_pd['adjt']
bart_pd['Net Rating'] = bart_pd['barthag'] * 100

# Map barttorvik names to TeamIDs
name_map_2026 = {}
bart_teams = bart_pd['team'].tolist()
pst_2025   = pst_pd[pst_pd['Season'] == 2025]
pst_teams  = pst_2025['Full Team Name'].tolist()
for bt in bart_teams:
    match, score, _ = process.extractOne(bt, pst_teams, scorer=fuzz.token_sort_ratio)
    name_map_2026[bt] = match if score >= 60 else None

manual_overrides = {
    "UConn": "Connecticut Huskies", "N.C. State": "NC State Wolfpack",
    "Miami FL": "Miami Hurricanes", "St. John's": "St. John's Red Storm",
    "Saint Mary's": "Saint Mary's Gaels", "VCU": "VCU Rams",
    "UCF": "UCF Knights", "UNLV": "UNLV Rebels", "UAB": "UAB Blazers",
    "BYU": "BYU Cougars", "LSU": "LSU Tigers", "SMU": "SMU Mustangs",
    "TCU": "TCU Horned Frogs", "McNeese": "McNeese State Cowboys",
    "Miami OH": "Miami (OH) RedHawks",
}
name_map_2026.update(manual_overrides)

# Build 2026 feature vectors indexed by TeamID
pst_avg_2025 = pst_2025[PST_FEATURES].mean()
team_id_features_2026 = {}

for _, row in bart_pd.iterrows():
    pst_name = name_map_2026.get(row['team'])
    if pst_name is None:
        continue
    tid = name_to_id(pst_name)
    if tid is None:
        continue
    
    # Start with 2025 averages for missing features
    feats = pst_avg_2025.to_dict()
    feats['AdjEM']      = row['AdjEM']
    feats['AdjOE']      = row['AdjOE']
    feats['AdjDE']      = row['AdjDE']
    feats['AdjTempo']   = row['AdjTempo']
    feats['Net Rating'] = row['Net Rating']
    
    # Add 2026 Massey rankings for this team
    massey_row = massey_2026[massey_2026['TeamID'] == tid]
    for mf in MASSEY_FEATURES:
        feats[mf] = massey_row[mf].values[0] if len(massey_row) > 0 and mf in massey_row.columns else 200
    
    # Add seed (will be filled per-matchup)
    feats['SeedNum'] = 8
    
    team_id_features_2026[tid] = feats

print(f"2026 teams with full features: {len(team_id_features_2026)}")

# ── STEP 8: ADD SEEDS TO 2026 FEATURE VECTORS ─────────────────────────────────
seeds_2026 = seeds_df.filter(pl.col('Season') == 2026).to_pandas()
seeds_2026['SeedNum'] = seeds_2026['Seed'].str[1:3].astype(int)

for _, row in seeds_2026.iterrows():
    tid = int(row['TeamID'])
    if tid in team_id_features_2026:
        team_id_features_2026[tid]['SeedNum'] = int(row['SeedNum'])

print("Seeds added to 2026 feature vectors!")

# ── STEP 9: ML PREDICTION FUNCTION ───────────────────────────────────────────
def ml_predict_winner(t1, t2):
    tid1 = int(t1['TeamID'])
    tid2 = int(t2['TeamID'])
    
    if tid1 not in team_id_features_2026 or tid2 not in team_id_features_2026:
        # Fallback to AdjEM
        return (t1, 0.6) if t1['AdjEM'] > t2['AdjEM'] else (t2, 0.6)
    
    f1 = team_id_features_2026[tid1]
    f2 = team_id_features_2026[tid2]
    
    diff = np.array([
        f1.get(f.replace('diff_', ''), 0) - f2.get(f.replace('diff_', ''), 0)
        for f in feature_cols
    ]).reshape(1, -1)
    
    prob_t1 = xgb_model.predict_proba(diff)[0][1]
    
    if prob_t1 >= 0.5:
        return t1, prob_t1
    else:
        return t2, 1 - prob_t1

# ── STEP 10: RUN FULL BRACKET ─────────────────────────────────────────────────

# Reload bracket with TeamIDs
seeds_2026_full = seeds_df.filter(pl.col('Season') == 2026).join(
    mteams.select(['TeamID', 'TeamName']), on='TeamID', how='left'
).to_pandas()
seeds_2026_full['Region']  = seeds_2026_full['Seed'].str[0]
seeds_2026_full['SeedNum'] = seeds_2026_full['Seed'].str[1:3].astype(int)

# Add AdjEM for display
def get_adjEM(team_name):
    match, score, _ = process.extractOne(team_name, bart_pd['team'].tolist(), scorer=fuzz.token_sort_ratio)
    return bart_pd[bart_pd['team'] == match]['AdjEM'].values[0] if score >= 60 else 0.0

seeds_2026_full['AdjEM'] = seeds_2026_full['TeamName'].apply(get_adjEM)

# Resolve play-in games
def play_in(seed_a, seed_b, winner_seed, df):
    t1 = df[df['Seed'] == seed_a].iloc[0]
    t2 = df[df['Seed'] == seed_b].iloc[0]
    winner, prob = ml_predict_winner(t1, t2)
    loser = t2 if winner['TeamName'] == t1['TeamName'] else t1
    print(f"  {t1['TeamName']:22s} vs {t2['TeamName']:22s}  →  ✓ {winner['TeamName']} ({prob:.0%})")
    new = winner.copy()
    new['Seed']    = winner_seed
    new['SeedNum'] = int(winner_seed[1:])
    return new, [seed_a, seed_b]

print(f"\n{'='*60}")
print("  FIRST FOUR (PLAY-IN GAMES)")
print(f"{'='*60}")

new_rows, remove = [], []
for sa, sb, ws in [('X16a','X16b','X16'), ('Y16a','Y16b','Y16'), ('Y11a','Y11b','Y11'), ('Z11a','Z11b','Z11')]:
    nr, rm = play_in(sa, sb, ws, seeds_2026_full)
    new_rows.append(nr)
    remove += rm

bracket_clean = seeds_2026_full[~seeds_2026_full['Seed'].isin(remove)].copy()
bracket_clean = pd.concat([bracket_clean, pd.DataFrame(new_rows)], ignore_index=True)

# Simulate regions
def simulate_region_ml(region_code, region_name):
    teams = bracket_clean[bracket_clean['Region'] == region_code].sort_values('SeedNum').reset_index(drop=True)
    
    print(f"\n{'='*60}")
    print(f"  REGION: {region_name}")
    print(f"{'='*60}")
    
    round_names = {1: 'Round of 64', 2: 'Round of 32', 3: 'Sweet 16', 4: 'Elite Eight'}
    round_num = 1
    
    while len(teams) > 1:
        print(f"\n  --- {round_names[round_num]} ---")
        winners = []
        n = len(teams)
        for i in range(n // 2):
            t1 = teams.iloc[i]
            t2 = teams.iloc[n - 1 - i]
            winner, prob = ml_predict_winner(t1, t2)
            print(f"  ({int(t1['SeedNum'])}) {t1['TeamName']:22s} vs ({int(t2['SeedNum'])}) {t2['TeamName']:22s}  →  ✓ {winner['TeamName']:22s} ({prob:.0%})")
            winners.append(winner)
        teams = pd.DataFrame(winners).reset_index(drop=True)
        round_num += 1
    
    print(f"\n  🏆 {region_name} Champion: {teams.iloc[0]['TeamName']}")
    return teams.iloc[0]

region_map   = {'W': 'West', 'X': 'East', 'Y': 'South', 'Z': 'Midwest'}
final_four   = []
for code, name in region_map.items():
    final_four.append(simulate_region_ml(code, name))

# Final Four
print(f"\n{'='*60}")
print("  FINAL FOUR")
print(f"{'='*60}")

ff = pd.DataFrame(final_four)
w  = ff[ff['Region'] == 'W'].iloc[0]
x  = ff[ff['Region'] == 'X'].iloc[0]
y  = ff[ff['Region'] == 'Y'].iloc[0]
z  = ff[ff['Region'] == 'Z'].iloc[0]

sf1, p1 = ml_predict_winner(w, x)
print(f"\n  (West)    {w['TeamName']:20s} vs (East)    {x['TeamName']:20s}  →  ✓ {sf1['TeamName']} ({p1:.0%})")

sf2, p2 = ml_predict_winner(y, z)
print(f"  (South)   {y['TeamName']:20s} vs (Midwest) {z['TeamName']:20s}  →  ✓ {sf2['TeamName']} ({p2:.0%})")

# Championship
print(f"\n{'='*60}")
print("  NATIONAL CHAMPIONSHIP")
print(f"{'='*60}")
champion, p_champ = ml_predict_winner(sf1, sf2)
print(f"\n  {sf1['TeamName']:20s} vs {sf2['TeamName']}")
print(f"\n  🏆 2026 NATIONAL CHAMPION: {champion['TeamName']} ({p_champ:.0%} confidence) 🏆")

In [0]:
# Check why so many matchups are being skipped
print("=== TRAINING DATA ISSUE ===")
print(f"Total tourney games: {len(tourney_pd)}")
print(f"Matched: {len(train_df)}, Skipped: {skipped}")

# Check a few specific teams that should be in the lookup
test_teams = [(2025, 1181), (2024, 1181), (2023, 1181)]  # Duke across seasons
for key in test_teams:
    print(f"Key {key} in lookup: {key in team_feat_lookup}")

# Check PST TeamID mapping success
print(f"\nPST TeamID nulls: {pst_pd['TeamID'].isna().sum()}")
print(f"PST TeamID sample:")
print(pst_pd[['Full Team Name', 'TeamID', 'Season']].head(10))

# Check 2026 feature lookup
print(f"\n2026 feature lookup size: {len(team_id_features_2026)}")
print(f"Sample TeamIDs in 2026 lookup: {list(team_id_features_2026.keys())[:10]}")

# Check bracket TeamIDs
print(f"\nSample bracket TeamIDs:")
print(bracket_clean[['TeamName', 'TeamID', 'SeedNum']].head(10))
print(f"TeamID dtype: {bracket_clean['TeamID'].dtype}")

In [0]:
# ── FIXED NAME TO ID MAPPING ──────────────────────────────────────────────────

# Build a better lookup using MTeams.csv TeamName directly
mteams_pd = mteams.to_pandas()

# Create multiple lookup dictionaries
spell_lookup = {}
for _, row in spell_pd.iterrows():
    spell_lookup[row['TeamNameSpelling'].lower().strip()] = int(row['TeamID'])

# Also add direct TeamName lookup from MTeams
for _, row in mteams_pd.iterrows():
    spell_lookup[row['TeamName'].lower().strip()] = int(row['TeamID'])

def name_to_id(full_name):
    if full_name is None or pd.isna(full_name):
        return None
    
    # Try exact match first
    key = str(full_name).lower().strip()
    if key in spell_lookup:
        return spell_lookup[key]
    
    # Try removing common suffixes (Blue Devils, Wildcats, etc.)
    # and matching just the school name
    words = key.split()
    for length in range(len(words), 0, -1):
        partial = ' '.join(words[:length])
        if partial in spell_lookup:
            return spell_lookup[partial]
    
    # Fuzzy match as last resort
    match, score, _ = process.extractOne(
        key, list(spell_lookup.keys()), scorer=fuzz.token_sort_ratio
    )
    return spell_lookup[match] if score >= 75 else None

# Test it
test_names = ['Duke Blue Devils', 'Kentucky Wildcats', 'UConn Huskies', 
              'Gonzaga Bulldogs', 'Auburn Tigers', 'Florida Gators']
for name in test_names:
    tid = name_to_id(name)
    print(f"  '{name}' → TeamID: {tid}")

In [0]:
# ── RE-MAP TeamIDs WITH FIXED FUNCTION ───────────────────────────────────────
pst_pd['TeamID'] = pst_pd['Full Team Name'].apply(name_to_id)
print(f"PST rows with TeamID mapped: {pst_pd['TeamID'].notna().sum()} / {len(pst_pd)}")

# Rebuild team feature lookup
pst_massey = pst_pd.merge(
    massey_wide,
    on=['Season', 'TeamID'],
    how='left'
).fillna(0)

team_feat_lookup = {}
for _, row in pst_massey.iterrows():
    if pd.notna(row['TeamID']):
        key = (int(row['Season']), int(row['TeamID']))
        team_feat_lookup[key] = {f: row.get(f, 0) for f in ALL_FEATURES}

print(f"Team-season feature vectors: {len(team_feat_lookup)}")

# Test Duke is now in lookup
print(f"Duke 2025 in lookup: {(2025, 1181) in team_feat_lookup}")
print(f"Duke 2024 in lookup: {(2024, 1181) in team_feat_lookup}")

In [0]:
# ── REBUILD TRAINING DATA ─────────────────────────────────────────────────────
records = []
skipped = 0

for _, row in tourney_pd.iterrows():
    season = int(row['Season'])
    w_team = int(row['WTeamID'])
    l_team = int(row['LTeamID'])
    
    team_a = min(w_team, l_team)
    team_b = max(w_team, l_team)
    label  = 1 if w_team == team_a else 0
    
    key_a = (season, team_a)
    key_b = (season, team_b)
    
    if key_a not in team_feat_lookup or key_b not in team_feat_lookup:
        skipped += 1
        continue
    
    fa = team_feat_lookup[key_a].copy()
    fb = team_feat_lookup[key_b].copy()
    
    try:
        fa['SeedNum'] = seed_lookup.loc[(season, team_a)]
        fb['SeedNum'] = seed_lookup.loc[(season, team_b)]
    except:
        fa['SeedNum'] = 8
        fb['SeedNum'] = 8
    
    diff = {f'diff_{f}': fa.get(f, 0) - fb.get(f, 0) for f in ALL_FEATURES}
    diff['label'] = label
    records.append(diff)

train_df     = pd.DataFrame(records)
feature_cols = [c for c in train_df.columns if c.startswith('diff_')]
print(f"\nReal tournament matchups: {len(train_df)}, Skipped: {skipped}")

In [0]:
# ── RETRAIN MODEL ─────────────────────────────────────────────────────────────
X = train_df[feature_cols].fillna(0).values
y = train_df['label'].values

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

lr = LogisticRegression(C=0.1, max_iter=1000, random_state=42)
lr_cv = cross_val_score(lr, X_scaled, y, cv=5, scoring='neg_log_loss')
print(f"Logistic Regression CV Log Loss: {-lr_cv.mean():.4f} ± {lr_cv.std():.4f}")

xgb_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='logloss', random_state=42
)
xgb_cv = cross_val_score(xgb_model, X, y, cv=5, scoring='neg_log_loss')
print(f"XGBoost CV Log Loss:             {-xgb_cv.mean():.4f} ± {xgb_cv.std():.4f}")

xgb_model.fit(X, y)
lr.fit(X_scaled, y)
print("Models retrained!")

# ── REBUILD 2026 FEATURE LOOKUP WITH FIXED MAPPING ───────────────────────────
team_id_features_2026 = {}
for _, row in bart_pd.iterrows():
    pst_name = name_map_2026.get(row['team'])
    if pst_name is None:
        continue
    tid = name_to_id(pst_name)
    if tid is None:
        continue
    
    feats = pst_avg_2025.to_dict()
    feats['AdjEM']      = row['AdjEM']
    feats['AdjOE']      = row['AdjOE']
    feats['AdjDE']      = row['AdjDE']
    feats['AdjTempo']   = row['AdjTempo']
    feats['Net Rating'] = row['Net Rating']
    
    massey_row = massey_2026[massey_2026['TeamID'] == tid]
    for mf in MASSEY_FEATURES:
        feats[mf] = massey_row[mf].values[0] if len(massey_row) > 0 and mf in massey_row.columns else 200
    
    feats['SeedNum'] = 8
    team_id_features_2026[tid] = feats

# Add seeds
for _, row in seeds_2026.iterrows():
    tid = int(row['TeamID'])
    if tid in team_id_features_2026:
        team_id_features_2026[tid]['SeedNum'] = int(row['SeedNum'])

print(f"2026 teams with features: {len(team_id_features_2026)}")

# Check key tournament teams are present
for name, tid in [('Duke', 1181), ('Michigan', 1272), ('Florida', 1199), ('Arizona', 1112)]:
    print(f"  {name} (TeamID {tid}): {'✓' if tid in team_id_features_2026 else '✗ MISSING'}")

In [0]:
# ── FIX 2026 TEAM MAPPING ─────────────────────────────────────────────────────

# Check what barttorvik names are mapping to Duke and Florida
print("Checking barttorvik name mappings for missing teams:")
for bart_name, pst_name in name_map_2026.items():
    if pst_name in ['Duke Blue Devils', 'Florida Gators', 'Connecticut Huskies']:
        tid = name_to_id(pst_name) if pst_name else None
        print(f"  bart='{bart_name}' → pst='{pst_name}' → TeamID={tid}")

# Check which tourney teams are still missing
seeds_2026_pd = seeds_df.filter(pl.col('Season') == 2026).join(
    mteams.select(['TeamID', 'TeamName']), on='TeamID', how='left'
).to_pandas()

missing_tids = []
for _, row in seeds_2026_pd.iterrows():
    tid = int(row['TeamID'])
    if tid not in team_id_features_2026:
        missing_tids.append((tid, row['TeamName']))

print(f"\nTournament teams missing 2026 features: {len(missing_tids)}")
for tid, name in missing_tids:
    print(f"  TeamID={tid}, TeamName='{name}'")

In [0]:
# ── DIRECT TEAMID OVERRIDES FOR MISSING TOURNAMENT TEAMS ─────────────────────
# Map barttorvik name directly to TeamID, bypassing pst name matching entirely

direct_tid_overrides = {
    "Duke":           1181,
    "UConn":          1163,
    "Kansas":         1242,
    "UCLA":           1417,
    "Ohio St.":       1326,
    "Cal Baptist":    1465,
    "Furman":         1202,
    "Siena":          1373,
    "Nebraska":       1304,
    "Iowa":           1234,
    "McNeese":        1270,
    "McNeese St.":    1270,
    "Troy":           1407,
    "Penn":           1335,
    "Idaho":          1225,
    "Lehigh":         1250,
    "Michigan":       1276,
    "Iowa St.":       1235,
    "Alabama":        1104,
    "Texas Tech":     1403,
    "UMBC":           1420,
    "Purdue":         1345,
    "Arkansas":       1116,
    "Miami FL":       1274,
    "Utah St.":       1429,
    "Texas":          1400,
    "Hawaii":         1218,
    "Queens NC":      1474,
    "LIU Brooklyn":   1254,
    "Florida":        1199,
    "Florida St.":    1198,
    "Connecticut":    1163,
    "Michigan St.":   1277,
    "St. John's":     1385,
    "Louisville":     1257,
    "TCU":            1395,
    "UCF":            1416,
    "South Florida":  1393,
    "Northern Iowa":  1314,
    "N. Iowa":        1314,
    "Houston":        1224,
    "Illinois":       1228,
    "Vanderbilt":     1426,
    "North Carolina": 1314,
    "UNC":            1314,
    "St. Mary's":     1366,
    "Saint Mary's":   1366,
    "Clemson":        1155,
    "Texas A&M":      1397,
    "VCU":            1433,
    "Virginia":       1438,
    "Iowa St.":       1235,
    "Tennessee":      1400,
    "Kentucky":       1246,
    "Santa Clara":    1360,
    "Georgia":        1207,
    "St. Louis":      1362,
    "Arizona":        1112,
    "Gonzaga":        1211,
    "Wisconsin":      1458,
    "BYU":            1137,
    "Missouri":       1295,
    "Villanova":      1437,
    "NC State":       1308,
    "N.C. State":     1308,
    "High Point":     1221,
    "Kennesaw":       1245,
    "Kennesaw St.":   1245,
    "Wright St.":     1462,
    "Wright St":      1462,
    "Tennessee St.":  1401,
    "Tennessee St":   1401,
    "Howard":         1226,
    "UMKC":           1430,
    "Prairie View":   1344,
    "SMU":            1382,
    "Miami OH":       1273,
    "Hofstra":        1222,
    "Akron":          1103,
    "McNeese St":     1270,
}

# ── REBUILD 2026 FEATURE LOOKUP USING DIRECT OVERRIDES ───────────────────────
team_id_features_2026 = {}

for _, row in bart_pd.iterrows():
    bart_name = row['team']
    
    # Try direct override first
    tid = direct_tid_overrides.get(bart_name)
    
    # Fall back to pst name mapping
    if tid is None:
        pst_name = name_map_2026.get(bart_name)
        if pst_name:
            tid = name_to_id(pst_name)
    
    if tid is None:
        continue
    
    feats = pst_avg_2025.to_dict()
    feats['AdjEM']      = row['AdjEM']
    feats['AdjOE']      = row['AdjOE']
    feats['AdjDE']      = row['AdjDE']
    feats['AdjTempo']   = row['AdjTempo']
    feats['Net Rating'] = row['Net Rating']
    
    massey_row = massey_2026[massey_2026['TeamID'] == tid]
    for mf in MASSEY_FEATURES:
        feats[mf] = massey_row[mf].values[0] if len(massey_row) > 0 and mf in massey_row.columns else 200
    
    feats['SeedNum'] = 8
    team_id_features_2026[tid] = feats

# Add seeds
for _, row in seeds_2026.iterrows():
    tid = int(row['TeamID'])
    if tid in team_id_features_2026:
        team_id_features_2026[tid]['SeedNum'] = int(row['SeedNum'])

print(f"2026 teams with features: {len(team_id_features_2026)}")

# Verify all tournament teams are now covered
still_missing = []
for _, row in seeds_2026_pd.iterrows():
    tid = int(row['TeamID'])
    if tid not in team_id_features_2026:
        still_missing.append((tid, row['TeamName']))

if still_missing:
    print(f"\nStill missing {len(still_missing)} teams:")
    for tid, name in still_missing:
        print(f"  TeamID={tid}, TeamName='{name}'")
else:
    print("\n✓ All tournament teams have 2026 features!")

In [0]:
# ── PROPER APPROACH: JOIN EVERYTHING ON TEAMID ───────────────────────────────
# 
# Flow:
# 1. Massey rankings → already has TeamID ✓
# 2. Seeds → already has TeamID ✓  
# 3. Tournament results → already has TeamID ✓
# 4. PST stats → needs name→TeamID mapping (one time fix)
# 5. Barttorvik 2026 → needs name→TeamID mapping (one time fix)
# Then everything joins cleanly on (Season, TeamID)

# ── BUILD MASTER TEAM LOOKUP FROM MTEAMS ─────────────────────────────────────
mteams_pd   = mteams.to_pandas()
spellings_pd = spellings.to_pandas()

# Full spelling lookup
spell_lookup = {}
for _, row in spellings_pd.iterrows():
    spell_lookup[row['TeamNameSpelling'].lower().strip()] = int(row['TeamID'])
for _, row in mteams_pd.iterrows():
    spell_lookup[row['TeamName'].lower().strip()] = int(row['TeamID'])

def name_to_id(full_name):
    if full_name is None or pd.isna(full_name):
        return None
    key = str(full_name).lower().strip()
    if key in spell_lookup:
        return spell_lookup[key]
    # Try progressively shorter versions
    words = key.split()
    for length in range(len(words), 0, -1):
        partial = ' '.join(words[:length])
        if partial in spell_lookup:
            return spell_lookup[partial]
    match, score, _ = process.extractOne(key, list(spell_lookup.keys()), scorer=fuzz.token_sort_ratio)
    return spell_lookup[match] if score >= 75 else None

# ── STEP 1: ADD TEAMID TO PST ─────────────────────────────────────────────────
pst_pd = pst.to_pandas()
pst_pd['TeamID'] = pst_pd['Full Team Name'].apply(name_to_id)
pst_pd['TeamID'] = pst_pd['TeamID'].astype('Int64')
print(f"PST mapped: {pst_pd['TeamID'].notna().sum()} / {len(pst_pd)}")

# ── STEP 2: ADD TEAMID TO BARTTORVIK 2026 ────────────────────────────────────
# Map barttorvik short names → TeamID using spellings
bart_pd = df_2026.to_pandas()
bart_pd['AdjEM']      = bart_pd['adjoe'] - bart_pd['adjde']
bart_pd['AdjOE']      = bart_pd['adjoe']
bart_pd['AdjDE']      = bart_pd['adjde']
bart_pd['AdjTempo']   = bart_pd['adjt']
bart_pd['Net Rating'] = bart_pd['barthag'] * 100

bart_pd['TeamID'] = bart_pd['team'].apply(name_to_id)
print(f"Barttorvik mapped: {bart_pd['TeamID'].notna().sum()} / {len(bart_pd)}")

# Check which tourney teams are still missing from barttorvik
seeds_2026_pd = seeds_df.filter(pl.col('Season') == 2026).join(
    mteams.select(['TeamID', 'TeamName']), on='TeamID', how='left'
).to_pandas()
seeds_2026_pd['SeedNum'] = seeds_2026_pd['Seed'].str[1:3].astype(int)
seeds_2026_pd['Region']  = seeds_2026_pd['Seed'].str[0]

tourney_tids = set(seeds_2026_pd['TeamID'].astype(int).tolist())
bart_tids    = set(bart_pd[bart_pd['TeamID'].notna()]['TeamID'].astype(int).tolist())
missing      = tourney_tids - bart_tids
print(f"\nTourney teams missing from barttorvik mapping: {len(missing)}")
for tid in sorted(missing):
    name = mteams_pd[mteams_pd['TeamID'] == tid]['TeamName'].values
    print(f"  TeamID={tid}, Name='{name[0] if len(name) else '?'}'")

In [0]:
# ── STEP 3: BUILD FEATURE TABLE USING TEAMID JOINS ───────────────────────────

PST_FEATURES = [f for f in [
    'AdjEM', 'AdjOE', 'AdjDE', 'AdjTempo', 'Net Rating',
    'eFGPct', 'TOPct', 'ORPct', 'FTRate', 'FG3Pct', 'FG3Rate',
    'BlockPct', 'OppFG3Pct', 'OppBlockPct', 'OppStlRate', 'StlRate',
    'Experience', 'AvgHeight', 'EffectiveHeight', 'Active Coaching Length Index',
] if f in pst_pd.columns]

MASSEY_FEATURES = [c for c in massey_wide.columns if c.startswith('Massey_')]
ALL_FEATURES    = PST_FEATURES + MASSEY_FEATURES + ['SeedNum']
print(f"Total features: {len(ALL_FEATURES)}")

# Join pst + massey on (Season, TeamID) — pure TeamID join, no name matching
pst_with_id = pst_pd[pst_pd['TeamID'].notna()].copy()
pst_with_id['TeamID'] = pst_with_id['TeamID'].astype(int)

pst_massey = pst_with_id.merge(
    massey_wide,
    on=['Season', 'TeamID'],
    how='left'
).fillna(0)

# Build lookup dict
team_feat_lookup = {}
for _, row in pst_massey.iterrows():
    key = (int(row['Season']), int(row['TeamID']))
    team_feat_lookup[key] = {f: float(row.get(f, 0)) for f in ALL_FEATURES}

# Add seeds to lookup
seeds_all = seeds_df.to_pandas()
seeds_all['SeedNum'] = seeds_all['Seed'].str[1:3].astype(int)
for _, row in seeds_all.iterrows():
    key = (int(row['Season']), int(row['TeamID']))
    if key in team_feat_lookup:
        team_feat_lookup[key]['SeedNum'] = int(row['SeedNum'])

print(f"Team-season vectors: {len(team_feat_lookup)}")
print(f"Duke 2025: {(2025, 1181) in team_feat_lookup}")
print(f"Duke 2024: {(2024, 1181) in team_feat_lookup}")

In [0]:
# ── STEP 4: BUILD TRAINING DATA FROM REAL TOURNEY GAMES ──────────────────────
tourney_pd = tourney_results.to_pandas()
tourney_pd = tourney_pd[tourney_pd['Season'] >= 2003]

records = []
skipped = 0

for _, row in tourney_pd.iterrows():
    season = int(row['Season'])
    w_team = int(row['WTeamID'])
    l_team = int(row['LTeamID'])
    team_a = min(w_team, l_team)
    team_b = max(w_team, l_team)
    label  = 1 if w_team == team_a else 0

    key_a = (season, team_a)
    key_b = (season, team_b)

    if key_a not in team_feat_lookup or key_b not in team_feat_lookup:
        skipped += 1
        continue

    fa = team_feat_lookup[key_a]
    fb = team_feat_lookup[key_b]
    diff = {f'diff_{f}': fa.get(f, 0) - fb.get(f, 0) for f in ALL_FEATURES}
    diff['label'] = label
    records.append(diff)

train_df     = pd.DataFrame(records)
feature_cols = [c for c in train_df.columns if c.startswith('diff_')]
print(f"Training matchups: {len(train_df)}, Skipped: {skipped}")

In [0]:
# ── STEP 5: TRAIN MODEL ───────────────────────────────────────────────────────
X = train_df[feature_cols].fillna(0).values
y = train_df['label'].values

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

lr = LogisticRegression(C=0.1, max_iter=1000, random_state=42)
lr_cv = cross_val_score(lr, X_scaled, y, cv=5, scoring='neg_log_loss')
print(f"Logistic Regression CV Log Loss: {-lr_cv.mean():.4f} ± {lr_cv.std():.4f}")

xgb_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='logloss', random_state=42
)
xgb_cv = cross_val_score(xgb_model, X, y, cv=5, scoring='neg_log_loss')
print(f"XGBoost CV Log Loss:             {-xgb_cv.mean():.4f} ± {xgb_cv.std():.4f}")

xgb_model.fit(X, y)
lr.fit(X_scaled, y)
print("Models trained!")

In [0]:
# ── STEP 6: BUILD 2026 FEATURES USING TEAMID JOIN ────────────────────────────
# Barttorvik already has TeamID mapped — just join with massey_2026

pst_avg_2025 = pst_pd[pst_pd['Season'] == 2025][PST_FEATURES].mean()

team_id_features_2026 = {}
bart_with_id = bart_pd[bart_pd['TeamID'].notna()].copy()
bart_with_id['TeamID'] = bart_with_id['TeamID'].astype(int)

# Join barttorvik with 2026 massey on TeamID
bart_massey_2026 = bart_with_id.merge(massey_2026, on='TeamID', how='left').fillna(200)

for _, row in bart_massey_2026.iterrows():
    tid  = int(row['TeamID'])
    feats = pst_avg_2025.to_dict()  # defaults for features not in barttorvik
    feats['AdjEM']      = row['AdjEM']
    feats['AdjOE']      = row['AdjOE']
    feats['AdjDE']      = row['AdjDE']
    feats['AdjTempo']   = row['AdjTempo']
    feats['Net Rating'] = row['Net Rating']
    for mf in MASSEY_FEATURES:
        feats[mf] = float(row.get(mf, 200))
    feats['SeedNum'] = 8
    team_id_features_2026[tid] = feats

# Add 2026 seeds
for _, row in seeds_2026_pd.iterrows():
    tid = int(row['TeamID'])
    if tid in team_id_features_2026:
        team_id_features_2026[tid]['SeedNum'] = int(row['SeedNum'])

print(f"2026 teams with features: {len(team_id_features_2026)}")

# Verify all tourney teams covered
still_missing = [(int(r['TeamID']), r['TeamName']) 
                 for _, r in seeds_2026_pd.iterrows() 
                 if int(r['TeamID']) not in team_id_features_2026]
if still_missing:
    print(f"Still missing {len(still_missing)}: {still_missing}")
else:
    print("✓ All tournament teams have 2026 features!")

In [0]:
# ── EXPLAIN HAWAII vs ARKANSAS PREDICTION ────────────────────────────────────

tid_hawaii   = 1218
tid_arkansas = 1116

f_hawaii   = team_id_features_2026[tid_hawaii]
f_arkansas = team_id_features_2026[tid_arkansas]

# Build the diff vector
diff = np.array([
    f_hawaii.get(f.replace('diff_', ''), 0) - f_arkansas.get(f.replace('diff_', ''), 0)
    for f in feature_cols
]).reshape(1, -1)

prob_hawaii = xgb_model.predict_proba(diff)[0][1]
print(f"Model probability — Hawaii wins: {prob_hawaii:.1%}")
print(f"Model probability — Arkansas wins: {1-prob_hawaii:.1%}")

# Show feature-by-feature comparison
print(f"\n{'Feature':<35} {'Hawaii':>10} {'Arkansas':>10} {'Diff (H-A)':>12}")
print("-" * 70)
for f in ALL_FEATURES:
    h_val = f_hawaii.get(f, 0)
    a_val = f_arkansas.get(f, 0)
    diff_val = h_val - a_val
    marker = " ◄" if abs(diff_val) > 5 else ""
    print(f"{f:<35} {h_val:>10.2f} {a_val:>10.2f} {diff_val:>12.2f}{marker}")

# Show SHAP-style feature importance for this specific matchup
print("\n\n=== TOP FEATURES DRIVING THIS PREDICTION ===")
import xgboost as xgb

# Get XGBoost feature importances
importances = xgb_model.feature_importances_
feat_imp = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': importances,
    'Diff': [
        f_hawaii.get(f.replace('diff_', ''), 0) - f_arkansas.get(f.replace('diff_', ''), 0)
        for f in feature_cols
    ]
}).sort_values('Importance', ascending=False)

feat_imp['Favors'] = feat_imp.apply(
    lambda r: f"Hawaii (+{abs(r['Diff']):.1f})" if r['Diff'] > 0 
              else f"Arkansas (+{abs(r['Diff']):.1f})", axis=1
)

print(f"\n{'Feature':<35} {'Importance':>10} {'Favors'}")
print("-" * 70)
for _, row in feat_imp.head(15).iterrows():
    print(f"{row['Feature']:<35} {row['Importance']:>10.4f}  {row['Favors']}")

# Also check raw barttorvik stats
print("\n\n=== RAW BARTTORVIK STATS ===")
hawaii_bart   = bart_pd[bart_pd['TeamID'] == tid_hawaii][['team','AdjEM','AdjOE','AdjDE','AdjTempo','barthag','WAB']].iloc[0]
arkansas_bart = bart_pd[bart_pd['TeamID'] == tid_arkansas][['team','AdjEM','AdjOE','AdjDE','AdjTempo','barthag','WAB']].iloc[0]
print(f"\n{'Stat':<15} {'Hawaii':>10} {'Arkansas':>10}")
print("-" * 38)
for col in ['AdjEM','AdjOE','AdjDE','AdjTempo','barthag','WAB']:
    print(f"{col:<15} {hawaii_bart[col]:>10.2f} {arkansas_bart[col]:>10.2f}")

In [0]:
# Check what TeamID Arkansas got mapped to
print("Arkansas in bart_pd:")
print(bart_pd[bart_pd['team'] == 'Arkansas'][['team', 'TeamID', 'AdjEM', 'AdjOE', 'AdjDE', 'barthag']])

print("\nWhat TeamID 1116 actually is in MTeams:")
print(mteams_pd[mteams_pd['TeamID'] == 1116])

print("\nWhat team has AdjEM=-10.79 in barttorvik:")
print(bart_pd[abs(bart_pd['AdjEM'] - (-10.79)) < 0.1][['team', 'TeamID', 'AdjEM']])

print("\nAlso check team_id_features_2026 for Arkansas (1116):")
print(team_id_features_2026.get(1116, {}).get('AdjEM'))
print(team_id_features_2026.get(1116, {}).get('AdjOE'))

# Check all tourney teams for suspicious AdjEM values
print("\n=== SANITY CHECK ALL TOURNEY TEAMS ===")
print(f"{'TeamName':<20} {'TeamID':>8} {'AdjEM':>8} {'AdjOE':>8} {'AdjDE':>8}")
print("-" * 55)
for _, row in seeds_2026_pd.drop_duplicates('TeamID').sort_values('SeedNum').iterrows():
    tid = int(row['TeamID'])
    if tid in team_id_features_2026:
        f = team_id_features_2026[tid]
        adjEM = f.get('AdjEM', 0)
        adjOE = f.get('AdjOE', 0)
        adjDE = f.get('AdjDE', 0)
        flag = " ◄ SUSPICIOUS" if adjEM < -5 or adjOE < 100 else ""
        print(f"{row['TeamName']:<20} {tid:>8} {adjEM:>8.2f} {adjOE:>8.2f} {adjDE:>8.2f}{flag}")

In [0]:
# ── FIX DUPLICATE TEAMID MAPPINGS ────────────────────────────────────────────

# These teams got overwritten by similarly-named teams in barttorvik
# Force correct stats directly from bart_pd

fixes = {
    1116: 'Arkansas',           # was overwritten by Arkansas Pine Bluff
    1397: 'Tennessee',          # was overwritten by Tennessee St/Martin
    1341: 'Prairie View',       # play-in team, low AdjEM is correct
    1250: 'Lehigh',             # play-in team, low AdjEM is correct  
    1254: 'LIU Brooklyn',       # low seed, low AdjEM is correct
}

for tid, bart_name in fixes.items():
    # Get the correct row from barttorvik
    bart_row = bart_pd[bart_pd['team'] == bart_name]
    if len(bart_row) == 0:
        print(f"WARNING: '{bart_name}' not found in barttorvik")
        continue
    
    bart_row = bart_row.iloc[0]
    print(f"Fixing {bart_name} (TeamID {tid}): AdjEM {team_id_features_2026[tid]['AdjEM']:.2f} → {bart_row['AdjEM']:.2f}")
    
    # Update the features
    team_id_features_2026[tid]['AdjEM']      = bart_row['AdjEM']
    team_id_features_2026[tid]['AdjOE']      = bart_row['AdjOE']
    team_id_features_2026[tid]['AdjDE']      = bart_row['AdjDE']
    team_id_features_2026[tid]['AdjTempo']   = bart_row['AdjTempo']
    team_id_features_2026[tid]['Net Rating'] = bart_row['Net Rating']
    
    # Also fix Massey rankings
    massey_row = massey_2026[massey_2026['TeamID'] == tid]
    for mf in MASSEY_FEATURES:
        team_id_features_2026[tid][mf] = massey_row[mf].values[0] if len(massey_row) > 0 and mf in massey_row.columns else 200

print("\n=== VERIFICATION ===")
check_teams = [(1116, 'Arkansas'), (1397, 'Tennessee')]
for tid, name in check_teams:
    f = team_id_features_2026[tid]
    print(f"{name} (TeamID {tid}): AdjEM={f['AdjEM']:.2f}, AdjOE={f['AdjOE']:.2f}, AdjDE={f['AdjDE']:.2f}")

# Also check Illinois which also looked suspicious (AdjEM=5.75 seems low for a 3 seed)
print(f"\nIllinois AdjEM in barttorvik:")
print(bart_pd[bart_pd['team'] == 'Illinois'][['team', 'TeamID', 'AdjEM', 'AdjOE', 'AdjDE']])
print(f"Illinois in team_id_features: AdjEM={team_id_features_2026.get(1228, {}).get('AdjEM', 'MISSING'):.2f}")

In [0]:
import shap
import matplotlib.pyplot as plt

# ── BUILD FULL FEATURE MATRIX FOR ALL 2026 TOURNEY MATCHUPS ──────────────────
# Create every possible tourney matchup so we can explain all predictions

tourney_tids = [int(r['TeamID']) for _, r in seeds_2026_pd.drop_duplicates('TeamID').iterrows()
                if int(r['TeamID']) in team_id_features_2026]

matchup_records = []
matchup_labels  = []

for i, tid1 in enumerate(tourney_tids):
    for tid2 in tourney_tids[i+1:]:
        f1 = team_id_features_2026[tid1]
        f2 = team_id_features_2026[tid2]
        diff = [f1.get(f.replace('diff_',''), 0) - f2.get(f.replace('diff_',''), 0)
                for f in feature_cols]
        matchup_records.append(diff)
        t1_name = seeds_2026_pd[seeds_2026_pd['TeamID'] == tid1]['TeamName'].values[0]
        t2_name = seeds_2026_pd[seeds_2026_pd['TeamID'] == tid2]['TeamName'].values[0]
        matchup_labels.append(f"{t1_name} vs {t2_name}")

X_2026 = np.array(matchup_records)
print(f"Total 2026 matchups to explain: {len(X_2026)}")

# ── SHAP EXPLAINER ────────────────────────────────────────────────────────────
explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_2026)

# Clean feature names for display
clean_names = [f.replace('diff_','').replace('Massey_','') for f in feature_cols]

print("SHAP values computed!")

In [0]:
# ── RUN FULL ML BRACKET ───────────────────────────────────────────────────────

def ml_predict_winner(t1, t2):
    tid1 = int(t1['TeamID'])
    tid2 = int(t2['TeamID'])

    if tid1 not in team_id_features_2026 or tid2 not in team_id_features_2026:
        return (t1, 0.6) if t1['AdjEM'] > t2['AdjEM'] else (t2, 0.6)

    f1 = team_id_features_2026[tid1]
    f2 = team_id_features_2026[tid2]

    # Always put lower TeamID first to match training convention
    if tid1 < tid2:
        team_lo, team_hi = t1, t2
        f_lo, f_hi       = f1, f2
    else:
        team_lo, team_hi = t2, t1
        f_lo, f_hi       = f2, f1

    diff = np.array([
        f_lo.get(f.replace('diff_', ''), 0) - f_hi.get(f.replace('diff_', ''), 0)
        for f in feature_cols
    ]).reshape(1, -1)

    # prob = P(lower TeamID team wins)
    prob_lo_wins = xgb_model.predict_proba(diff)[0][1]

    if prob_lo_wins >= 0.5:
        return team_lo, prob_lo_wins       # lower ID team wins
    else:
        return team_hi, 1 - prob_lo_wins   # higher ID team wins

# Add AdjEM to bracket for display
def get_adjEM(tid):
    tid = int(tid)
    return team_id_features_2026[tid]['AdjEM'] if tid in team_id_features_2026 else 0.0

seeds_2026_pd['AdjEM'] = seeds_2026_pd['TeamID'].apply(get_adjEM)

# ── RESOLVE PLAY-IN GAMES ─────────────────────────────────────────────────────
print(f"{'='*60}")
print("  FIRST FOUR (PLAY-IN GAMES)")
print(f"{'='*60}")

def play_in(seed_a, seed_b, winner_seed, df):
    t1 = df[df['Seed'] == seed_a].iloc[0]
    t2 = df[df['Seed'] == seed_b].iloc[0]
    winner, prob = ml_predict_winner(t1, t2)
    print(f"  {t1['TeamName']:22s} vs {t2['TeamName']:22s}  →  ✓ {winner['TeamName']} ({prob:.0%})")
    new = winner.copy()
    new['Seed']    = winner_seed
    new['SeedNum'] = int(winner_seed[1:])
    return new, [seed_a, seed_b]

new_rows, remove = [], []
for sa, sb, ws in [('X16a','X16b','X16'), ('Y16a','Y16b','Y16'),
                   ('Y11a','Y11b','Y11'), ('Z11a','Z11b','Z11')]:
    nr, rm = play_in(sa, sb, ws, seeds_2026_pd)
    new_rows.append(nr)
    remove += rm

bracket_clean = seeds_2026_pd[~seeds_2026_pd['Seed'].isin(remove)].copy()
bracket_clean = pd.concat([bracket_clean, pd.DataFrame(new_rows)], ignore_index=True)

# Verify 16 per region
for r in ['W','X','Y','Z']:
    n = len(bracket_clean[bracket_clean['Region'] == r])
    print(f"Region {r}: {n} teams {'✓' if n==16 else '✗'}")

# ── SIMULATE REGIONS ──────────────────────────────────────────────────────────
def simulate_region(region_code, region_name):
    teams = bracket_clean[bracket_clean['Region'] == region_code].sort_values('SeedNum').reset_index(drop=True)

    print(f"\n{'='*60}")
    print(f"  REGION: {region_name}")
    print(f"{'='*60}")

    round_names = {1:'Round of 64', 2:'Round of 32', 3:'Sweet 16', 4:'Elite Eight'}
    round_num = 1

    while len(teams) > 1:
        print(f"\n  --- {round_names[round_num]} ---")
        winners = []
        n = len(teams)
        for i in range(n // 2):
            t1 = teams.iloc[i]
            t2 = teams.iloc[n - 1 - i]
            winner, prob = ml_predict_winner(t1, t2)
            print(f"  ({int(t1['SeedNum'])}) {t1['TeamName']:22s} vs ({int(t2['SeedNum'])}) {t2['TeamName']:22s}  →  ✓ {winner['TeamName']:22s} ({prob:.0%})")
            winners.append(winner)
        teams = pd.DataFrame(winners).reset_index(drop=True)
        round_num += 1

    print(f"\n  🏆 {region_name} Champion: {teams.iloc[0]['TeamName']}")
    return teams.iloc[0]

region_map = {'W':'West', 'X':'East', 'Y':'South', 'Z':'Midwest'}
final_four = []
for code, name in region_map.items():
    final_four.append(simulate_region(code, name))

# ── FINAL FOUR ────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("  FINAL FOUR")
print(f"{'='*60}")

ff = pd.DataFrame(final_four)
w  = ff[ff['Region'] == 'W'].iloc[0]
x  = ff[ff['Region'] == 'X'].iloc[0]
y  = ff[ff['Region'] == 'Y'].iloc[0]
z  = ff[ff['Region'] == 'Z'].iloc[0]

sf1, p1 = ml_predict_winner(w, x)
print(f"\n  (West)  {w['TeamName']:20s} vs (East)    {x['TeamName']:20s}  →  ✓ {sf1['TeamName']} ({p1:.0%})")

sf2, p2 = ml_predict_winner(y, z)
print(f"  (South) {y['TeamName']:20s} vs (Midwest) {z['TeamName']:20s}  →  ✓ {sf2['TeamName']} ({p2:.0%})")

# ── CHAMPIONSHIP ──────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("  NATIONAL CHAMPIONSHIP")
print(f"{'='*60}")

champion, p_champ = ml_predict_winner(sf1, sf2)
print(f"\n  {sf1['TeamName']:20s} vs {sf2['TeamName']}")
print(f"\n  🏆 2026 NATIONAL CHAMPION: {champion['TeamName']} ({p_champ:.0%} confidence) 🏆")

In [0]:
# ── PLOT 1: GLOBAL FEATURE IMPORTANCE (SHAP) ─────────────────────────────────
# Shows which features matter most across ALL matchups

plt.figure(figsize=(12, 8))
shap.summary_plot(
    shap_values, 
    X_2026, 
    feature_names=clean_names,
    plot_type="bar",
    show=False,
    max_display=20
)
plt.title("Feature Importance — Mean |SHAP Value| Across All 2026 Matchups", fontsize=14)
plt.tight_layout()
plt.savefig("shap_importance.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved shap_importance.png")

In [0]:
# ── PLOT 2: SHAP BEE SWARM — shows direction + magnitude ─────────────────────
# Red = high feature value pushes prediction UP (team 1 wins)
# Blue = low feature value pushes prediction DOWN

plt.figure(figsize=(12, 10))
shap.summary_plot(
    shap_values,
    X_2026,
    feature_names=clean_names,
    show=False,
    max_display=20
)
plt.title("SHAP Beeswarm — Feature Impact on Win Probability", fontsize=14)
plt.tight_layout()
plt.savefig("shap_beeswarm.png", dpi=150, bbox_inches='tight')
plt.show()

In [0]:
# ── PLOT 3: EXPLAIN A SPECIFIC MATCHUP ───────────────────────────────────────
# Walk through exactly why the model predicted what it did for key games

def explain_matchup(team1_name, team2_name):
    # Find TeamIDs
    t1_row = seeds_2026_pd[seeds_2026_pd['TeamName'] == team1_name]
    t2_row = seeds_2026_pd[seeds_2026_pd['TeamName'] == team2_name]
    
    if len(t1_row) == 0 or len(t2_row) == 0:
        print(f"Team not found — check spelling")
        return
    
    tid1 = int(t1_row.iloc[0]['TeamID'])
    tid2 = int(t2_row.iloc[0]['TeamID'])
    
    f1 = team_id_features_2026[tid1]
    f2 = team_id_features_2026[tid2]
    
    diff = np.array([
        f1.get(f.replace('diff_',''), 0) - f2.get(f.replace('diff_',''), 0)
        for f in feature_cols
    ]).reshape(1, -1)
    
    prob = xgb_model.predict_proba(diff)[0][1]
    
    print(f"\n{'='*60}")
    print(f"  EXPLAINING: {team1_name} vs {team2_name}")
    print(f"  Model says: {team1_name} wins with {prob:.1%} probability")
    print(f"{'='*60}")
    
    # SHAP for this specific matchup
    shap_single = explainer.shap_values(diff)[0]
    
    # Build explanation table
    explanation = pd.DataFrame({
        'Feature':    clean_names,
        'Team1_Val':  [f1.get(f.replace('diff_',''), 0) for f in feature_cols],
        'Team2_Val':  [f2.get(f.replace('diff_',''), 0) for f in feature_cols],
        'Difference': diff[0],
        'SHAP':       shap_single
    }).sort_values('SHAP', key=abs, ascending=False)
    
    print(f"\n{'Feature':<25} {team1_name:>12} {team2_name:>12} {'Diff':>8} {'SHAP Impact':>12} {'Favors'}")
    print("-" * 85)
    for _, row in explanation.head(15).iterrows():
        favors = team1_name if row['SHAP'] > 0 else team2_name
        bar    = "█" * int(abs(row['SHAP']) * 50)
        print(f"  {row['Feature']:<23} {row['Team1_Val']:>12.2f} {row['Team2_Val']:>12.2f} "
              f"{row['Difference']:>8.2f} {row['SHAP']:>12.4f}  {favors} {bar}")
    
    # Waterfall plot
    shap_exp = shap.Explanation(
        values        = shap_single,
        base_values   = explainer.expected_value,
        data          = diff[0],
        feature_names = clean_names
    )
    plt.figure(figsize=(12, 7))
    shap.waterfall_plot(shap_exp, max_display=15, show=False)
    plt.title(f"SHAP Waterfall: {team1_name} vs {team2_name} ({prob:.0%})", fontsize=13)
    plt.tight_layout()
    plt.savefig(f"shap_{team1_name.replace(' ','_')}_vs_{team2_name.replace(' ','_')}.png", 
                dpi=150, bbox_inches='tight')
    plt.show()

# Explain your key matchups
explain_matchup("Duke", "Connecticut")        # Elite Eight West
explain_matchup("Houston", "Florida")         # Elite Eight East upset
explain_matchup("Arizona", "Michigan")        # Final Four
explain_matchup("Duke", "Arizona")            # Championship

In [0]:
# ── PLOT 5: TRAINING DATA SHAP — what drove historical predictions ─────────────
# This shows what the model learned from 2003-2025 tournament games

shap_train   = explainer.shap_values(X)
clean_names_train = [f.replace('diff_','').replace('Massey_','') for f in feature_cols]

plt.figure(figsize=(12, 8))
shap.summary_plot(
    shap_train,
    X,
    feature_names=clean_names_train,
    show=False,
    max_display=20
)
plt.title("SHAP on Training Data (2003-2025 Tournament Games)", fontsize=14)
plt.tight_layout()
plt.savefig("shap_training.png", dpi=150, bbox_inches='tight')
plt.show()

# Print top 10 most important features numerically
mean_shap = pd.DataFrame({
    'Feature':   clean_names_train,
    'Mean_SHAP': np.abs(shap_train).mean(axis=0)
}).sort_values('Mean_SHAP', ascending=False)

print("\nTop 10 Features by Mean |SHAP| on Training Data:")
print(mean_shap.head(10).to_string(index=False))

In [0]:
def ml_predict_winner(t1, t2):
    tid1 = int(t1['TeamID'])
    tid2 = int(t2['TeamID'])

    if tid1 not in team_id_features_2026 or tid2 not in team_id_features_2026:
        return (t1, 0.6) if t1['AdjEM'] > t2['AdjEM'] else (t2, 0.6)

    f1 = team_id_features_2026[tid1]
    f2 = team_id_features_2026[tid2]

    # Always put lower TeamID first to match training convention
    if tid1 < tid2:
        team_lo, team_hi = t1, t2
        f_lo, f_hi       = f1, f2
    else:
        team_lo, team_hi = t2, t1
        f_lo, f_hi       = f2, f1

    diff = np.array([
        f_lo.get(f.replace('diff_', ''), 0) - f_hi.get(f.replace('diff_', ''), 0)
        for f in feature_cols
    ]).reshape(1, -1)

    # prob = P(lower TeamID team wins)
    prob_lo_wins = xgb_model.predict_proba(diff)[0][1]

    if prob_lo_wins >= 0.5:
        return team_lo, prob_lo_wins       # lower ID team wins
    else:
        return team_hi, 1 - prob_lo_wins   # higher ID team wins

# Also fix explain_matchup to use same convention
def explain_matchup(team1_name, team2_name):
    t1_row = seeds_2026_pd[seeds_2026_pd['TeamName'] == team1_name]
    t2_row = seeds_2026_pd[seeds_2026_pd['TeamName'] == team2_name]

    if len(t1_row) == 0 or len(t2_row) == 0:
        print(f"Team not found — check spelling")
        return

    tid1 = int(t1_row.iloc[0]['TeamID'])
    tid2 = int(t2_row.iloc[0]['TeamID'])

    f1 = team_id_features_2026[tid1]
    f2 = team_id_features_2026[tid2]

    # Match training convention: lower TeamID first
    if tid1 < tid2:
        tid_lo, tid_hi   = tid1, tid2
        name_lo, name_hi = team1_name, team2_name
        f_lo, f_hi       = f1, f2
    else:
        tid_lo, tid_hi   = tid2, tid1
        name_lo, name_hi = team2_name, team1_name
        f_lo, f_hi       = f2, f1

    diff = np.array([
        f_lo.get(f.replace('diff_', ''), 0) - f_hi.get(f.replace('diff_', ''), 0)
        for f in feature_cols
    ]).reshape(1, -1)

    prob_lo = xgb_model.predict_proba(diff)[0][1]
    prob_hi = 1 - prob_lo

    winner = name_lo if prob_lo >= 0.5 else name_hi
    prob_w = max(prob_lo, prob_hi)

    print(f"\n{'='*60}")
    print(f"  EXPLAINING: {team1_name} vs {team2_name}")
    print(f"  Lower TeamID team: {name_lo} (TeamID {tid_lo})")
    print(f"  P({name_lo} wins) = {prob_lo:.1%}")
    print(f"  P({name_hi} wins) = {prob_hi:.1%}")
    print(f"  ✓ Model picks: {winner} ({prob_w:.1%})")
    print(f"{'='*60}")

    shap_single = explainer.shap_values(diff)[0]

    explanation = pd.DataFrame({
        'Feature':    clean_names,
        'LowID_Val':  [f_lo.get(f.replace('diff_',''), 0) for f in feature_cols],
        'HighID_Val': [f_hi.get(f.replace('diff_',''), 0) for f in feature_cols],
        'Difference': diff[0],
        'SHAP':       shap_single
    }).sort_values('SHAP', key=abs, ascending=False)

    print(f"\n{'Feature':<25} {name_lo:>14} {name_hi:>14} {'Diff':>8} {'SHAP':>10} {'Favors'}")
    print("-" * 90)
    for _, row in explanation.head(15).iterrows():
        favors = name_lo if row['SHAP'] > 0 else name_hi
        bar    = "█" * int(abs(row['SHAP']) * 50)
        print(f"  {row['Feature']:<23} {row['LowID_Val']:>14.2f} {row['HighID_Val']:>14.2f} "
              f"{row['Difference']:>8.2f} {row['SHAP']:>10.4f}  {favors} {bar}")

    shap_exp = shap.Explanation(
        values        = shap_single,
        base_values   = explainer.expected_value,
        data          = diff[0],
        feature_names = clean_names
    )
    plt.figure(figsize=(12, 7))
    shap.waterfall_plot(shap_exp, max_display=15, show=False)
    plt.title(f"SHAP Waterfall: {team1_name} vs {team2_name}\n"
              f"P({name_lo} wins)={prob_lo:.0%}, P({name_hi} wins)={prob_hi:.0%}", fontsize=13)
    plt.tight_layout()
    plt.savefig(f"shap_{team1_name.replace(' ','_')}_vs_{team2_name.replace(' ','_')}.png",
                dpi=150, bbox_inches='tight')
    plt.show()

# Re-explain key matchups with correct convention
explain_matchup("Duke", "Connecticut")
explain_matchup("Houston", "Florida")
explain_matchup("Arizona", "Michigan")
explain_matchup("Duke", "Arizona")